In [5]:
# building a sample vector database
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

```markdown
1. Gerekli Kütüphaneleri Yükleme
Amaç: Vektör veritabanı oluşturmak, belge yüklemek, embedding ve metin bölme işlemleri için gerekli kütüphaneleri eklemek.
Kütüphaneler:
Chroma: Vektör veritabanı sağlar.
TextLoader: Düz metin dosyasını belgeye çevirir.
OllamaEmbeddings: Metinleri vektörlere dönüştürür.
RecursiveCharacterTextSplitter: Metni parçalara böler.
```

In [6]:
loader = TextLoader("speech.txt")
data = loader.load()
data

[Document(metadata={'source': 'speech.txt'}, page_content='    I have called the Congress into extraordinary session because there are serious, very serious, choices of policy to be made, and made immediately, which it was neither right nor constitutionally permissible that I should assume the responsibility of making. On the 3rd of February last, I officially laid before you the extraordinary announcement of the Imperial German government that on and after the 1st day of February it was its purpose to put aside all restraints of law or of humanity and use its submarines to sink every vessel that sought to approach either the ports of Great Britain and Ireland or the western coasts of Europe or any of the ports controlled by the enemies of Germany within the Mediterranean...\n\n    When I addressed the Congress on the 26th of February last, I thought that it would suffice to assert our neutral rights with arms, our right to use the seas against unlawful interference, our right to keep 

```markdown
2. Metin Dosyasını Yükleme
Amaç: speech.txt dosyasını belge olarak yüklemek.

```

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
splits = text_splitter.split_documents(data)

```markdown
3. Metni Parçalara Bölme
Amaç: Metni 500 karakterlik parçalara ayırmak.
Kütüphane: RecursiveCharacterTextSplitter
```

In [8]:
embedding = OllamaEmbeddings(model="gemma:2b")
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding
)
vectordb

C:\Users\murat\AppData\Local\Temp\ipykernel_16916\3552194669.py:1: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import OllamaEmbeddings``.
  embedding = OllamaEmbeddings(model="gemma:2b")


```markdown
4. Embedding Modeli ve Chroma Vektör Veritabanı Oluşturma
Amaç: Parçalanmış belgeleri embedding vektörlerine dönüştürüp Chroma vektör veritabanında saklamak.
Kütüphaneler: OllamaEmbeddings, Chroma
```

In [9]:
# query
query = "What is the main topic of the speech?"
results = vectordb.similarity_search(query, k=3)
for result in results:
    print(result.page_content)
    print("-----")



prevent; it is practically certain to draw us into the war without either the rights or the effectiveness of belligerents. There is one choice we cannot make, we are incapable of making: we will not choose the path of submission and suffer the most sacred rights of our nation and our people to be ignored or violated. The wrongs against which we now array ourselves are no common wrongs; they cut to the very roots of human life.
-----
The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them...
-----
immediate steps, not only to put the country in a more thorough state of defense but

```markdown
5. Sorgu ile Benzer Belgeleri Arama
Amaç: Sorguya en yakın 3 belgeyi bulmak.

```

In [11]:
# Save the vector store to disk
vectordb = Chroma.from_documents(
    documents=splits,
    embedding=embedding,
    persist_directory="./chroma_db_gemma2b"  
)

```markdown
6. Vektör Veritabanını Diske Kaydetme
Amaç: Chroma vektör veritabanını diske kaydetmek.

```

In [13]:
# Load from disk
vectordb = Chroma(persist_directory="./chroma_db_gemma2b", embedding_function=embedding)
# Query the loaded vector store
results = vectordb.similarity_search(query, k=3)
for result in results:
    print(result.page_content)
    print("-----")

prevent; it is practically certain to draw us into the war without either the rights or the effectiveness of belligerents. There is one choice we cannot make, we are incapable of making: we will not choose the path of submission and suffer the most sacred rights of our nation and our people to be ignored or violated. The wrongs against which we now array ourselves are no common wrongs; they cut to the very roots of human life.
-----
The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them...
-----
immediate steps, not only to put the country in a more thorough state of defense but

```markdown
7. Kaydedilen Veritabanını Yükleme ve Sorgulama
Amaç: Kaydedilmiş Chroma veritabanını tekrar yükleyip sorgu yapmak.

```

In [14]:
# Similiarity search with scores
results_with_scores = vectordb.similarity_search_with_score(query, k=3)
for result, score in results_with_scores:
    print(f"Score: {score}")
    print(result.page_content)
    print("-----")

Score: 2963.97021484375
prevent; it is practically certain to draw us into the war without either the rights or the effectiveness of belligerents. There is one choice we cannot make, we are incapable of making: we will not choose the path of submission and suffer the most sacred rights of our nation and our people to be ignored or violated. The wrongs against which we now array ourselves are no common wrongs; they cut to the very roots of human life.
-----
Score: 3352.627197265625
The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them...
-----
Score: 3590.13818359375
immediate s

```markdown
8. Skorlarla Benzerlik Araması
Amaç: Benzer belgeleri ve benzerlik skorlarını birlikte görmek.

```

In [16]:
# Retriever
retriever = vectordb.as_retriever(search_type="similarity", search_kwargs={"k": 3})
retriever.invoke(query)[0].page_content



'prevent; it is practically certain to draw us into the war without either the rights or the effectiveness of belligerents. There is one choice we cannot make, we are incapable of making: we will not choose the path of submission and suffer the most sacred rights of our nation and our people to be ignored or violated. The wrongs against which we now array ourselves are no common wrongs; they cut to the very roots of human life.'

```markdown
9. Retriever ile Sorgu
Amaç: Retriever arayüzüyle benzer belgeleri almak.

```

```markdown
Genel Amaç
Bu notebook'ta amaç, metin belgelerini embedding vektörlerine dönüştürüp Chroma ile hızlı benzerlik araması yapmaktır. Belgeler parçalara ayrılır, embedding ile vektörleştirilir, Chroma veritabanında saklanır ve sorgularla benzer belgeler bulunabilir. Ayrıca veritabanı kaydedilip tekrar yüklenebilir.


```